# 38b. Two registered secondary outputs of notebook 38 (computed after the evaluation)

Here I compute two outputs that the notebook 38 analysis plan (OSF 10.17605/OSF.IO/BKJUQ) and `38_SPEC.json` list as secondary but that the notebook 38 code did not produce:
1. the per-day AUC decomposition pooled over indices (`per_day.pooled_over_indices: secondary`);
2. the event-level test and bootstrap with a 7-day linking gap for market episodes (`event_level.episodes.secondary_gap_calendar_days: 7`).

These are registered secondary outputs that I compute after the notebook 38 evaluation (2026-11-03 09:39 UTC); they have no decision rules and do not change the Q1–Q4 readings.
I write the results only to `external_equity_indices_v1b_secondary`.
With the frozen notebook 38 core code, specification, raw data, models and thresholds, I rerun the notebook 38 phase-2 code with the same seeds and compute the two secondary outputs from that rerun.
I count the 7-day episode statistics from the same null draws, using random numbers in the same order as in notebook 38.

In [1]:
from pathlib import Path
import os, sys, json, time, hashlib, importlib.util
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
TEST_DIR = os.environ.get('NB38B_TEST_OUT38')          # synthetic-data test only
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive') / 'btc_lead_cost_results_77months_spot'
except ImportError:
    BASE = Path(os.environ.get('NB38_BASE', '.'))
OUT38 = Path(TEST_DIR) if TEST_DIR else BASE / 'external_equity_indices_v1'          # read only
OUTB = (OUT38.parent / (OUT38.name + '_38b_TEST')) if TEST_DIR else BASE / 'external_equity_indices_v1b_secondary'
OUTB.mkdir(parents=True, exist_ok=True)
assert OUTB.resolve() != OUT38.resolve()
OUT = OUT38          # the reused notebook-38 code reads its frozen files from OUT

EXPECTED_SPEC_SHA256 = '994ec994626d023acf90241d3c90c3d4baaa78c930e2cea0a22c00571f3e5481'
EXPECTED_CORE_SHA256 = '273f0f7df0f5ac939db8015a211a94272eb75c5c107b9ebb8795ee465dab04ea'
EXPECTED_FROZEN_SHA256 = 'edab20d2dd110fe93fe2a0623eba1761e88d4460db234718fd528d3ff998e50a'    # from the notebook-38 lock


def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()


SPEC_BYTES = (OUT38 / '38_SPEC.json').read_bytes(); SPEC_SHA = hashlib.sha256(SPEC_BYTES).hexdigest(); SPEC = json.loads(SPEC_BYTES)
if SPEC_SHA != EXPECTED_SPEC_SHA256 or sha(OUT38 / '38_core.py') != EXPECTED_CORE_SHA256:
    raise RuntimeError('GUARD FAIL: 38_SPEC.json or 38_core.py differs from the registered version')
LOCK38 = json.loads((OUT38 / 'EVALUATED.lock').read_text())
if not TEST_DIR and LOCK38['frozen_sha256'] != EXPECTED_FROZEN_SHA256:
    raise RuntimeError('GUARD FAIL: notebook-38 lock does not match the evaluated run')
if sha(OUT38 / 'FROZEN.json') != LOCK38['frozen_sha256']:
    raise RuntimeError('GUARD FAIL: FROZEN.json differs from the one locked by notebook 38')
_spec = importlib.util.spec_from_file_location('core38', OUT38 / '38_core.py'); C = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(C)
MAN = OUT38 / 'raw_manifest.json'; manifest = json.loads(MAN.read_text()); RAW = OUT38 / 'raw'
for tk, m in manifest['indices'].items():
    if m.get('file') and sha(RAW / m['file']) != m['sha256']:
        raise RuntimeError(f'GUARD FAIL: raw file of {tk} changed since download')
INDICES = [tk for tk, m in manifest['indices'].items() if m['included']]
CLOSE = {tk: pd.read_csv(RAW / manifest['indices'][tk]['file'], index_col=0, parse_dates=True)['close'] for tk in INDICES}
print('notebook-38 lock:', LOCK38['evaluated_utc'], '| SPEC, core, FROZEN and raw files verified | indices:', len(INDICES))

Mounted at /content/drive
notebook-38 lock: 2026-11-03T09:39:18.018952+00:00 | SPEC, core, FROZEN and raw files verified | indices: 10


## 1. Frozen notebook 38 models and thresholds (no training)

In [2]:
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

T_CAL0, T_CAL1 = (pd.Timestamp(x) for x in SPEC['periods']['calibration'])
T_EV0, T_EV1 = (pd.Timestamp(x) for x in SPEC['periods']['evaluation'])
EVD = SPEC['events']; WIN = SPEC['window']; BUD = SPEC['budget']
SCHED = dict(q_cap=BUD['quarter_cap'], y_cap=BUD['year_cap'], cooldown=BUD['cooldown_days'])
FEAT = {tk: C.features(CLOSE[tk]) for tk in INDICES}
VALID = {tk: FEAT[tk][C.FEATS].notna().all(axis=1).to_numpy() for tk in INDICES}
FROZEN_P = OUT / 'FROZEN.json'

if FROZEN_P.exists():
    FROZEN = json.loads(FROZEN_P.read_text())
    for k in ['M0', 'GBM']:
        if C.sha256_bytes((OUT / FROZEN['models'][k]['file']).read_bytes()) != FROZEN['models'][k]['sha256']:
            raise RuntimeError(f'GUARD FAIL: frozen {k} model changed')
    if FROZEN['spec_sha256'] != SPEC_SHA or FROZEN['raw_manifest_sha256'] != C.sha256_bytes(MAN.read_bytes()):
        raise RuntimeError('GUARD FAIL: frozen models were made under a different SPEC or raw data')
    print('phase 1 already frozen', FROZEN['frozen_utc'])
else:
    Xs, ys, ntr = [], [], {}
    for tk in INDICES:
        c = CLOSE[tk]; pre = c[c.index < T_CAL0]
        E_tr = C.catalog(pre, EVD['main_H'], EVD['theta'], EVD['refractory_days'], EVD['onset_fraction_of_theta'])
        lab, _ = C.window_labels(len(pre), E_tr, WIN['wmin'], WIN['wmax'])
        keep = VALID[tk][:len(pre)].copy(); keep[max(0, len(pre) - SPEC['periods']['training_drop_last_days']):] = False
        Xs.append(FEAT[tk].iloc[:len(pre)][C.FEATS].to_numpy(float)[keep]); ys.append((lab[keep] > 0).astype(int))
        ntr[tk] = dict(rows=int(keep.sum()), positives=int((lab[keep] > 0).sum()), events=int(len(E_tr)))
    X = np.vstack(Xs); y = np.concatenate(ys)
    fi = [C.FEATS.index('z_vol'), C.FEATS.index('ret_5')]
    m0 = make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400, solver='lbfgs', random_state=17)).fit(X[:, fi], y)
    gbm = HistGradientBoostingClassifier(**SPEC['systems']['GBM']['params']).fit(X.astype(np.float32), y)
    models = {}
    for k, m in [('M0', m0), ('GBM', gbm)]:
        p = OUT / f'model_{k}.joblib'; joblib.dump(m, p); models[k] = dict(file=p.name, sha256=C.sha256_bytes(p.read_bytes()))
    thr = {}
    for tk in INDICES:
        d = FEAT[tk].index; cal = VALID[tk] & (d >= T_CAL0) & (d < T_CAL1); pos = np.flatnonzero(cal)
        Xc = FEAT[tk][C.FEATS].to_numpy(float)[cal]
        sc = {'M0': m0.predict_proba(Xc[:, fi])[:, 1], 'M1': FEAT[tk]['z_vol'].to_numpy(float)[cal], 'GBM': gbm.predict_proba(Xc.astype(np.float32))[:, 1]}
        thr[tk] = {k: dict(zip(['threshold', 'cal_alerts'], C.calibrate(pos, d[cal], v, SCHED))) for k, v in sc.items()}
    FROZEN = dict(frozen_utc=pd.Timestamp.now(tz='UTC').isoformat(), spec_sha256=SPEC_SHA, raw_manifest_sha256=C.sha256_bytes(MAN.read_bytes()),
                  models=models, thresholds=thr, training=ntr, training_rows=int(len(y)), training_positive_share=float(y.mean()))
    FROZEN_P.write_text(json.dumps(FROZEN, indent=2))
    print(f'phase 1 frozen: {len(y):,} training days ({y.mean():.1%} positive)')
FROZEN_SHA = C.sha256_bytes(FROZEN_P.read_bytes())
M0 = joblib.load(OUT / FROZEN['models']['M0']['file']); GBM = joblib.load(OUT / FROZEN['models']['GBM']['file'])
display(pd.DataFrame({tk: {k: v['cal_alerts'] for k, v in FROZEN['thresholds'][tk].items()} for tk in INDICES}).T.rename(columns=lambda c: f'{c} calibration alerts'))

phase 1 already frozen 2026-11-03T09:25:01.244309+00:00


,M0 calibration alerts,M1 calibration alerts,GBM calibration alerts
^GSPC,20,20,20
^IXIC,20,20,20
^GSPTSE,20,20,20
^FTSE,20,20,20
^GDAXI,20,20,20
^FCHI,20,20,20
^SSMI,20,20,20
^N225,20,20,20
^HSI,20,20,20
^AXJO,20,20,20


## 2. Rerun of notebook 38 phase 2 with the same code and seeds

In [3]:
T0 = time.time()
SYS_ALERT = ['M0', 'M1', 'GBM']; SYS_ALL = ['M0', 'M1', 'GBM', 'B0', 'B1']
fi = [C.FEATS.index('z_vol'), C.FEATS.index('ret_5')]
QBASE = T_EV0.year * 4 + (T_EV0.month - 1) // 3
NQ = (T_EV1.year * 4 + (T_EV1.month - 1) // 3) - QBASE

D = {}
for tk in INDICES:
    c = CLOSE[tk]; d = c.index; n = len(c)
    ev = VALID[tk] & (d >= T_EV0) & (d < T_EV1); evpos = np.flatnonzero(ev)
    Xe = FEAT[tk][C.FEATS].to_numpy(float)
    S = {k: np.full(n, np.nan) for k in SYS_ALL}
    S['M0'][ev] = M0.predict_proba(Xe[ev][:, fi])[:, 1]; S['GBM'][ev] = GBM.predict_proba(Xe[ev].astype(np.float32))[:, 1]
    S['M1'][ev] = FEAT[tk]['z_vol'].to_numpy(float)[ev]; S['B0'][ev] = FEAT[tk]['B0'].to_numpy(float)[ev]; S['B1'][ev] = FEAT[tk]['B1'].to_numpy(float)[ev]
    A = {k: C.schedule(evpos, d[ev], S[k][ev] >= FROZEN['thresholds'][tk][k]['threshold'], **SCHED) for k in SYS_ALERT}
    cats = {}
    for H in [EVD['main_H']] + EVD['secondary_H']:
        E = C.catalog(c, H, EVD['theta'], EVD['refractory_days'], EVD['onset_fraction_of_theta'])
        E = E[(E.breach_pos >= evpos[0] + SPEC['periods']['event_must_breach_after_eval_start_days']) & (E.breach_date < T_EV1)].reset_index(drop=True)
        cats[H] = E
    quarter = np.full(n, -1, np.int64); dq = d.year.to_numpy() * 4 + (d.month.to_numpy() - 1) // 3 - QBASE; quarter[ev] = dq[ev]
    rvdec = np.full(n, -1, np.int64); rvdec[ev] = np.minimum((pd.Series(S['B0'][ev]).rank(pct=True).to_numpy() * 10).astype(int), 9)
    D[tk] = dict(n=n, dates=d, ev=ev, evpos=evpos, S=S, A=A, E=cats, quarter=quarter, rvdec=rvdec)
print('alerts:', {k: sum(len(D[tk]['A'][k]) for tk in INDICES) for k in SYS_ALERT},
      '| events:', {H: sum(len(D[tk]['E'][H]) for tk in INDICES) for H in D[INDICES[0]]['E']})

# ---- event and alert lists (dates and positions; no index levels)
rows = []
for tk in INDICES:
    for H, E in D[tk]['E'].items():
        e = E.copy(); e.insert(0, 'H', H); e.insert(0, 'index', tk); rows.append(e)
pd.concat(rows, ignore_index=True).to_csv(OUTB / '1_events.csv', index=False)
pd.DataFrame([dict(index=tk, system=k, alert_date=D[tk]['dates'][p].date(), alert_pos=int(p)) for tk in INDICES for k in SYS_ALERT for p in D[tk]['A'][k]]).to_csv(OUTB / '1b_alerts.csv', index=False)

# ---- E2 feasibility (labels only)
rows = []
for H in [10, 20, 60]:
    for wm in SPEC['feasibility']['wmax']:
        L = np.concatenate([C.feasibility(D[tk]['E'][H], wm, WIN['wmin']) for tk in INDICES]) if H in D[INDICES[0]]['E'] else np.array([])
        rows.append(dict(H=H, wmax=wm, events=len(L), impossible=int((L == 0).sum()), F=float((L > 0).mean()) if len(L) else np.nan))
T2 = pd.DataFrame(rows); T2.to_csv(OUTB / '2_feasibility.csv', index=False)
bad = T2[(T2.H <= T2.wmax) & (T2.F < 1)]
if len(bad):
    raise RuntimeError(f'GUARD FAIL (Corollary 1): {bad.to_dict("records")}')
display(T2.pivot(index='H', columns='wmax', values='F').round(3))

alerts: {'M0': 805, 'M1': 823, 'GBM': 713} | events: {20: 241, 10: 146, 60: 233}


wmax,5,10,20,40,60
H,,,,,
10,0.315,1.000,1.000,1.000,1.0
20,0.120,0.398,1.000,1.000,1.0
60,0.047,0.112,0.386,0.747,1.0


In [4]:
# ---- E1/E3 per-day AUCs with a common calendar-quarter bootstrap
rng = np.random.default_rng(SPEC['per_day']['bootstrap']['seed'])
NB = SPEC['per_day']['bootstrap']['draws']
WQ = np.vstack([np.ones(NQ)] + [np.bincount(rng.integers(0, NQ, NQ), minlength=NQ).astype(float) for _ in range(NB)])   # row 0 = point estimate


def preps(H, negset='main', with_reg=True):
    """Per index and system: AUC preparations for standard, pre-onset, in-progress and within-regime pre-onset."""
    P = {}
    for tk in INDICES:
        x = D[tk]; lab, _ = C.window_labels(x['n'], x['E'][H], WIN['wmin'], WIN['wmax'])
        keep = x['ev'] & np.all([np.isfinite(x['S'][k]) for k in SYS_ALL], axis=0)
        neg = keep & (lab == 0)
        if negset == 'secondary':
            for b in x['E'][H].breach_pos:
                neg[b:b + WIN['wmax'] + 1] = False
        P[tk] = {}
        for k in SYS_ALL:
            s = x['S'][k]
            P[tk][k] = dict(std=C.AucPrep(s, lab > 0, neg | (keep & (lab > 0)), x['quarter']),
                            pre=C.AucPrep(s, lab == 1, neg | (keep & (lab == 1)), x['quarter']),
                            inp=C.AucPrep(s, lab == 2, neg | (keep & (lab == 2)), x['quarter']),
                            reg=C.StratAucPrep(s, lab == 1, neg | (keep & (lab == 1)), x['quarter'], x['rvdec']) if with_reg else None,
                            q_in=x['quarter'][keep & (lab == 2)], q_pos=x['quarter'][keep & (lab > 0)])
    return P


def evaluate(P, w):
    """Equal-weighted averages over indices for one quarter-weight vector w."""
    out = {}
    for k in SYS_ALL:
        a = {m: [] for m in ['std', 'pre', 'inp', 'reg']}; num, den, pis = [], [], []
        for tk in INDICES:
            p = P[tk][k]; v = {m: (p[m].auc(w) if p[m] is not None else np.nan) for m in ['std', 'pre', 'inp', 'reg']}
            wpos = w[p['q_pos']].sum(); pi = w[p['q_in']].sum() / wpos if wpos > 0 else np.nan
            for m in a:
                a[m].append(v[m])
            if np.isfinite(v['std']) and np.isfinite(v['inp']) and np.isfinite(pi):
                num.append(pi * (v['inp'] - 0.5)); den.append(v['std'] - 0.5); pis.append(pi)
        r = {m: (float(np.nanmean(a[m])) if np.isfinite(a[m]).any() else np.nan) for m in a}
        r['pi'] = float(np.mean(pis)) if pis else np.nan; r['excess'] = float(np.mean(den)) if den else np.nan
        r['S_in'] = float(np.mean(num) / np.mean(den)) if den and np.mean(den) != 0 else np.nan
        out[k] = r
    return out


BOOT = {}
for H, negset in [(20, 'main'), (60, 'main'), (10, 'main'), (20, 'secondary'), (60, 'secondary')]:
    t_ = time.time(); P = preps(H, negset, with_reg=(H == 20 and negset == 'main'))
    BOOT[(H, negset)] = [evaluate(P, WQ[i]) for i in range(NB + 1)]
    print(f'per-day bootstrap H = {H}, negatives {negset}: {time.time() - t_:.0f}s')

rows = []
for (H, negset), draws in BOOT.items():
    for k in SYS_ALL:
        for m in ['std', 'pre', 'inp', 'reg', 'pi', 'excess', 'S_in']:
            v = np.array([d_[k][m] for d_ in draws[1:]])
            rows.append(dict(H=H, negatives=negset, system=k, measure=m, point=draws[0][k][m],
                             lo95=float(np.nanpercentile(v, 2.5)), hi95=float(np.nanpercentile(v, 97.5)), finite_draws=int(np.isfinite(v).sum())))
    for m in ['std', 'pre', 'inp', 'reg']:
        v = np.array([d_['GBM'][m] - d_['M0'][m] for d_ in draws[1:]])
        rows.append(dict(H=H, negatives=negset, system='GBM - M0', measure=m, point=draws[0]['GBM'][m] - draws[0]['M0'][m],
                         lo95=float(np.nanpercentile(v, 2.5)), hi95=float(np.nanpercentile(v, 97.5)), finite_draws=int(np.isfinite(v).sum())))
T1 = pd.DataFrame(rows); T1.to_csv(OUTB / '3_per_day_auc.csv', index=False)
display(T1[(T1.H == 20) & (T1.negatives == 'main') & T1.measure.isin(['std', 'pre', 'inp', 'reg', 'pi', 'S_in'])].pivot(index='system', columns='measure', values='point').round(3))

# per-index point values (descriptive)
P20 = preps(20); rows = []
for tk in INDICES:
    for k in SYS_ALL:
        p = P20[tk][k]; w = np.ones(NQ)
        rows.append(dict(index=tk, system=k, events=len(D[tk]['E'][20]), std=p['std'].auc(w), pre=p['pre'].auc(w), inp=p['inp'].auc(w), reg=p['reg'].auc(w),
                         pi=len(p['q_in']) / max(len(p['q_pos']), 1)))
pd.DataFrame(rows).to_csv(OUTB / '3b_per_day_auc_by_index.csv', index=False)

per-day bootstrap H = 20, negatives main: 76s
per-day bootstrap H = 60, negatives main: 41s
per-day bootstrap H = 10, negatives main: 40s
per-day bootstrap H = 20, negatives secondary: 37s
per-day bootstrap H = 60, negatives secondary: 38s


/usr/local/lib/python3.13/dist-packages/numpy/lib/_nanfunctions_impl.py:1396: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


measure,S_in,inp,pi,pre,reg,std
system,,,,,,
B0,0.629,0.770,0.537,0.684,0.567,0.730
B1,0.843,0.873,0.537,0.579,0.503,0.737
GBM,0.756,0.841,0.537,0.626,0.534,0.742
GBM - M0,NaN,0.018,NaN,0.038,0.037,0.027
M0,0.807,0.824,0.537,0.589,0.497,0.715
M1,0.589,0.700,0.537,0.662,0.541,0.683


In [5]:
# ---- E4 event level (D20): observed catches, three nulls, market episodes, episode bootstrap
H = EVD['main_H']
EV = pd.concat([D[tk]['E'][H].assign(index=tk) for tk in INDICES], ignore_index=True)
SL = {tk: np.flatnonzero(EV['index'].to_numpy() == tk) for tk in INDICES}
G_MAIN = SPEC['event_level']['episodes']['link_gap_calendar_days']; G_SEC = SPEC['event_level']['episodes']['secondary_gap_calendar_days']
EPI = C.episodes(EV.breach_date, G_MAIN); EPI2 = C.episodes(EV.breach_date, G_SEC); NEP = int(EPI.max()) + 1
EV['episode'] = EPI; EV['episode_secondary'] = EPI2
EV['episode_indices'] = EV.groupby('episode')['index'].transform('nunique')


def catches(alerts_by_index):
    pre = np.zeros(len(EV), bool); hit = np.zeros(len(EV), bool); al = np.full(len(EV), -1, np.int64)
    for tk in INDICES:
        h, p, a = C.match(D[tk]['E'][H], alerts_by_index[tk], WIN['wmin'], WIN['wmax'])
        hit[SL[tk]] = h; pre[SL[tk]] = p; al[SL[tk]] = a
    return hit, pre, al


OBS = {}
for k in SYS_ALERT:
    hit, pre, al = catches({tk: D[tk]['A'][k] for tk in INDICES})
    OBS[k] = dict(hit=hit, pre=pre, inp=hit & ~pre, alert=al, alerts=sum(len(D[tk]['A'][k]) for tk in INDICES))
    EV[f'{k}_pre'] = pre.astype(int); EV[f'{k}_in'] = (hit & ~pre).astype(int)
    EV[f'{k}_alert_minus_onset_days'] = np.where(hit, al - EV.onset_pos.to_numpy(), np.nan)

span = (T_EV1 - T_EV0).days
QPOS = {tk: C.quarter_positions(D[tk]['evpos'], D[tk]['quarter']) for tk in INDICES}
rngN = np.random.default_rng(SPEC['event_level']['seed_nulls'])
NULLS = {'budget-preserving': SPEC['event_level']['nulls']['budget_preserving_regime']['draws'],
         'joint circular': SPEC['event_level']['nulls']['joint_circular']['draws'],
         'short joint shift': SPEC['event_level']['nulls']['short_joint_shift']['draws']}
NR = {}; NSTAT = {k: {} for k in SYS_ALERT}
for k in SYS_ALERT:
    for nname, n_ in NULLS.items():
        t_ = time.time(); p_pre = np.zeros(len(EV)); p_in = np.zeros(len(EV))
        cnt = {m: np.empty(n_, np.int64) for m in ['pre', 'inp', 'ep_pre', 'ep_inp', 'ep2_pre', 'ep2_inp']}
        for i in range(n_):
            if nname == 'budget-preserving':
                sh = {tk: C.null_budget_preserving(rngN, D[tk]['A'][k], QPOS[tk], D[tk]['quarter'], D[tk]['rvdec'], BUD['cooldown_days'], NSTAT[k]) for tk in INDICES}
            elif nname == 'joint circular':
                sh = C.null_shift({tk: D[tk]['A'][k] for tk in INDICES}, {tk: D[tk]['dates'] for tk in INDICES}, SPEC['periods']['evaluation'], int(rngN.integers(365, span - 365)))
            else:
                sh = C.null_shift({tk: D[tk]['A'][k] for tk in INDICES}, {tk: D[tk]['dates'] for tk in INDICES}, SPEC['periods']['evaluation'],
                                  int(rngN.integers(28, 85)) * int(rngN.choice([-1, 1])))
            hit, pre, _ = catches(sh); inp = hit & ~pre
            p_pre += pre; p_in += inp
            cnt['pre'][i] = pre.sum(); cnt['inp'][i] = inp.sum(); cnt['ep_pre'][i] = np.unique(EPI[pre]).size; cnt['ep_inp'][i] = np.unique(EPI[inp]).size
            cnt['ep2_pre'][i] = np.unique(EPI2[pre]).size; cnt['ep2_inp'][i] = np.unique(EPI2[inp]).size
        NR[(k, nname)] = dict(n=n_, cnt=cnt, p_pre=p_pre / n_, p_in=p_in / n_)
        print(f'{k:>3} {nname:<18} {n_:>5} draws ({time.time() - t_:.0f}s)')

rows = []
for k in SYS_ALERT:
    for nname in NULLS:
        x = NR[(k, nname)]
        for m, obs in [('pre', OBS[k]['pre'].sum()), ('inp', OBS[k]['inp'].sum()),
                       ('ep_pre', np.unique(EPI[OBS[k]['pre']]).size), ('ep_inp', np.unique(EPI[OBS[k]['inp']]).size)]:
            v = x['cnt'][m]
            rows.append(dict(system=k, null=nname, statistic=m, observed=int(obs), null_mean=float(v.mean()), null_q05=float(np.percentile(v, 5)),
                             null_q95=float(np.percentile(v, 95)), p_one_sided=(1 + int((v >= obs).sum())) / (len(v) + 1), draws=len(v),
                             events=len(EV), episodes=NEP, alerts=OBS[k]['alerts']))
T4 = pd.DataFrame(rows)
for m in ['ep_pre', 'ep_inp']:
    sel = (T4['null'] == 'budget-preserving') & (T4.statistic == m)
    T4.loc[sel, 'p_holm'] = C.holm(T4.loc[sel, 'p_one_sided'].to_numpy())
T4.to_csv(OUTB / '4_event_level_tests.csv', index=False)
pd.DataFrame(NSTAT).T.fillna(0).astype(int).to_csv(OUTB / '4b_null_placement_diagnostics.csv')
print('budget-preserving placements:', NSTAT)
display(T4[T4['null'] == 'budget-preserving'].round(4))

rngB = np.random.default_rng(SPEC['event_level']['episode_bootstrap']['seed']); NBE = SPEC['event_level']['episode_bootstrap']['draws']
IDX = rngB.integers(0, NEP, size=(NBE, NEP))
rows = []; DIFF = {}
for k in SYS_ALERT:
    for nname in NULLS:
        x = NR[(k, nname)]; al = OBS[k]['alerts']
        ex_pre = np.bincount(EPI, OBS[k]['pre'] - x['p_pre'], minlength=NEP); ex_in = np.bincount(EPI, OBS[k]['inp'] - x['p_in'], minlength=NEP)
        b_pre = 100 * ex_pre[IDX].sum(1) / al; b_in = 100 * ex_in[IDX].sum(1) / al; b_d = b_in - b_pre
        for m, pt, b in [('pre', 100 * ex_pre.sum() / al, b_pre), ('inp', 100 * ex_in.sum() / al, b_in), ('inp - pre', 100 * (ex_in.sum() - ex_pre.sum()) / al, b_d)]:
            rows.append(dict(system=k, null=nname, measure=m, excess_per_100_alerts=float(pt), lo95=float(np.percentile(b, 2.5)), hi95=float(np.percentile(b, 97.5)),
                             alerts=al, episodes=NEP, draws=NBE))
        if nname == 'budget-preserving':
            DIFF[k] = (float(np.percentile(b_d, 2.5)), float(np.percentile(b_d, 97.5)))
T5 = pd.DataFrame(rows); T5.to_csv(OUTB / '5_effect_sizes.csv', index=False)
display(T5[T5['null'] == 'budget-preserving'].round(3))

size = EV['episode_indices'].to_numpy(); rows = []
for k in SYS_ALERT:
    x = NR[(k, 'budget-preserving')]
    for lo_, hi_, lab_ in [(1, 1, '1 index'), (2, 4, '2-4 indices'), (5, 99, '5+ indices'), (1, 99, 'all')]:
        sel = (size >= lo_) & (size <= hi_)
        for m, o, p in [('pre', OBS[k]['pre'], x['p_pre']), ('inp', OBS[k]['inp'], x['p_in'])]:
            rows.append(dict(system=k, endpoint=m, episode_size=lab_, events=int(sel.sum()), observed=int(o[sel].sum()), expected=float(p[sel].sum())))
pd.DataFrame(rows).to_csv(OUTB / '5b_excess_by_episode_size.csv', index=False)

rows = []
for k in SYS_ALERT:
    for m in ['pre', 'inp']:
        v = EV.loc[OBS[k][m].astype(bool), f'{k}_alert_minus_onset_days'].to_numpy()
        rows.append(dict(system=k, endpoint=m, n=len(v), **({f'alert_minus_onset_p{q}': float(np.percentile(v, q)) for q in (10, 25, 50, 75, 90)} if len(v) else {})))
pd.DataFrame(rows).to_csv(OUTB / '6_lead_times.csv', index=False)
EV.drop(columns=[]).to_csv(OUTB / '1c_event_level.csv', index=False)

 M0 budget-preserving   5000 draws (179s)
 M0 joint circular      5000 draws (35s)
 M0 short joint shift   1000 draws (8s)
 M1 budget-preserving   5000 draws (184s)
 M1 joint circular      5000 draws (37s)
 M1 short joint shift   1000 draws (7s)
GBM budget-preserving   5000 draws (159s)
GBM joint circular      5000 draws (35s)
GBM short joint shift   1000 draws (8s)
budget-preserving placements: {'M0': {'own decile': 2579134, 'other decile': 1445244, 'kept original': 622}, 'M1': {'other decile': 1412312, 'own decile': 2702688}, 'GBM': {'own decile': 2151967, 'other decile': 1412697, 'kept original': 336}}


,system,null,statistic,observed,null_mean,null_q05,null_q95,p_one_sided,draws,events,episodes,alerts,p_holm
0,M0,budget-preserving,pre,21,45.7768,40.0,51.0,1.0000,5000,241,52,805,NaN
1,M0,budget-preserving,inp,78,62.1400,56.0,68.0,0.0004,5000,241,52,805,NaN
2,M0,budget-preserving,ep_pre,9,20.8454,18.0,23.0,1.0000,5000,241,52,805,1.0000
3,M0,budget-preserving,ep_inp,28,30.4678,27.0,33.0,0.9490,5000,241,52,805,1.0000
12,M1,budget-preserving,pre,32,49.3702,44.0,55.0,1.0000,5000,241,52,823,NaN
13,M1,budget-preserving,inp,49,62.0384,56.0,68.0,1.0000,5000,241,52,823,NaN
14,M1,budget-preserving,ep_pre,13,20.4778,18.0,23.0,1.0000,5000,241,52,823,1.0000
15,M1,budget-preserving,ep_inp,20,27.9290,25.0,31.0,1.0000,5000,241,52,823,1.0000
24,GBM,budget-preserving,pre,30,31.2228,26.0,37.0,0.7031,5000,241,52,713,NaN
25,GBM,budget-preserving,inp,85,53.3378,48.0,59.0,0.0002,5000,241,52,713,NaN


,system,null,measure,excess_per_100_alerts,lo95,hi95,alerts,episodes,draws
0,M0,budget-preserving,pre,-3.078,-6.346,0.178,805,52,10000
1,M0,budget-preserving,inp,1.970,-2.826,6.959,805,52,10000
2,M0,budget-preserving,inp - pre,5.048,-0.582,11.412,805,52,10000
9,M1,budget-preserving,pre,-2.111,-5.555,1.293,823,52,10000
10,M1,budget-preserving,inp,-1.584,-5.402,2.595,823,52,10000
11,M1,budget-preserving,inp - pre,0.526,-4.561,6.209,823,52,10000
18,GBM,budget-preserving,pre,-0.172,-3.019,2.822,713,52,10000
19,GBM,budget-preserving,inp,4.441,-0.268,9.479,713,52,10000
20,GBM,budget-preserving,inp - pre,4.612,-0.952,10.659,713,52,10000


## 3. Comparison with the stored notebook 38 results

In [6]:
FILES = ['1_events.csv', '1b_alerts.csv', '1c_event_level.csv', '2_feasibility.csv', '3_per_day_auc.csv', '3b_per_day_auc_by_index.csv',
         '4_event_level_tests.csv', '4b_null_placement_diagnostics.csv', '5_effect_sizes.csv', '5b_excess_by_episode_size.csv', '6_lead_times.csv']
bad = []
for f in FILES:
    a = pd.read_csv(OUT38 / f); b = pd.read_csv(OUTB / f)
    if a.shape != b.shape or list(a.columns) != list(b.columns):
        bad.append((f, 'shape/columns')); continue
    num = a.select_dtypes('number').columns
    d = np.nanmax(np.abs(a[num].to_numpy(float) - b[num].to_numpy(float))) if len(num) else 0.0
    nan_ok = (a[num].isna().to_numpy() == b[num].isna().to_numpy()).all() if len(num) else True
    txt = [c for c in a.columns if c not in num]
    txt_ok = all((a[c].fillna('NA').astype(str) == b[c].fillna('NA').astype(str)).all() for c in txt)
    if not (d <= 1e-9 and nan_ok and txt_ok):
        bad.append((f, float(d), bool(nan_ok), bool(txt_ok)))
if bad:
    raise RuntimeError(f'GUARD FAIL: the re-run does not reproduce notebook 38: {bad}')
print(f'reproduction guard ok: {len(FILES)} notebook-38 output files reproduced exactly')

reproduction guard ok: 11 notebook-38 output files reproduced exactly


## 4. Secondary output 1: per-day AUC decomposition pooled over indices

In [7]:
def pooled_parts(H, negset):
    Sa = {k: [] for k in SYS_ALL}; LAB, Q, STR = [], [], []
    for j, tk in enumerate(INDICES):
        x = D[tk]; lab, _ = C.window_labels(x['n'], x['E'][H], WIN['wmin'], WIN['wmax'])
        keep = x['ev'] & np.all([np.isfinite(x['S'][k]) for k in SYS_ALL], axis=0)
        neg = keep & (lab == 0)
        if negset == 'secondary':
            for b in x['E'][H].breach_pos:
                neg[b:b + WIN['wmax'] + 1] = False
        idx = np.flatnonzero(neg | (keep & (lab > 0)))
        for k in SYS_ALL:
            Sa[k].append(x['S'][k][idx])
        LAB.append(lab[idx]); Q.append(x['quarter'][idx]); STR.append(j * 10 + x['rvdec'][idx])
    lab = np.concatenate(LAB); q = np.concatenate(Q); st = np.concatenate(STR); allk = np.ones(len(lab), bool)
    P = {}
    for k in SYS_ALL:
        s = np.concatenate(Sa[k])
        P[k] = dict(std=C.AucPrep(s, lab > 0, allk, q), pre=C.AucPrep(s, lab == 1, lab != 2, q), inp=C.AucPrep(s, lab == 2, lab != 1, q),
                    reg=C.StratAucPrep(s, lab == 1, lab != 2, q, st) if (H == 20 and negset == 'main') else None)
    return P, q[lab == 2], q[lab > 0]


def evaluate_pooled(P, qin, qpos, w):
    out = {}
    wpos = w[qpos].sum(); pi = w[qin].sum() / wpos if wpos > 0 else np.nan
    for k in SYS_ALL:
        r = {m: (P[k][m].auc(w) if P[k][m] is not None else np.nan) for m in ['std', 'pre', 'inp', 'reg']}
        r['pi'] = pi; r['excess'] = r['std'] - 0.5
        r['S_in'] = pi * (r['inp'] - 0.5) / (r['std'] - 0.5) if r['std'] != 0.5 else np.nan
        out[k] = r
    return out


rows = []
for H_, negset in [(20, 'main'), (60, 'main'), (10, 'main'), (20, 'secondary'), (60, 'secondary')]:
    t_ = time.time(); P, qin, qpos = pooled_parts(H_, negset)
    draws = [evaluate_pooled(P, qin, qpos, WQ[i]) for i in range(NB + 1)]
    for k in SYS_ALL:
        for m in ['std', 'pre', 'inp', 'reg', 'pi', 'excess', 'S_in']:
            v = np.array([d_[k][m] for d_ in draws[1:]], float)
            if not np.isfinite(v).any():
                continue
            rows.append(dict(H=H_, negatives=negset, system=k, measure=m, point=draws[0][k][m],
                             lo95=float(np.nanpercentile(v, 2.5)), hi95=float(np.nanpercentile(v, 97.5)), finite_draws=int(np.isfinite(v).sum())))
    for m in ['std', 'pre', 'inp', 'reg']:
        v = np.array([d_['GBM'][m] - d_['M0'][m] for d_ in draws[1:]], float)
        if np.isfinite(v).any():
            rows.append(dict(H=H_, negatives=negset, system='GBM - M0', measure=m, point=draws[0]['GBM'][m] - draws[0]['M0'][m],
                             lo95=float(np.nanpercentile(v, 2.5)), hi95=float(np.nanpercentile(v, 97.5)), finite_draws=int(np.isfinite(v).sum())))
    print(f'pooled per-day bootstrap H = {H_}, negatives {negset}: {time.time() - t_:.0f}s')
T7 = pd.DataFrame(rows); T7.to_csv(OUTB / '7_pooled_per_day_auc.csv', index=False)
display(T7[(T7.H == 20) & (T7.negatives == 'main')].pivot(index='system', columns='measure', values='point').round(3))

pooled per-day bootstrap H = 20, negatives main: 68s
pooled per-day bootstrap H = 60, negatives main: 32s
pooled per-day bootstrap H = 10, negatives main: 32s
pooled per-day bootstrap H = 20, negatives secondary: 29s
pooled per-day bootstrap H = 60, negatives secondary: 29s


measure,S_in,excess,inp,pi,pre,reg,std
system,,,,,,,
B0,0.619,0.242,0.778,0.538,0.699,0.561,0.742
B1,0.848,0.237,0.873,0.538,0.578,0.498,0.737
GBM,0.739,0.251,0.845,0.538,0.641,0.539,0.751
GBM - M0,NaN,NaN,0.027,NaN,0.068,0.048,0.046
M0,0.834,0.205,0.817,0.538,0.573,0.491,0.705
M1,0.590,0.171,0.688,0.538,0.652,0.537,0.671


## 5. Secondary output 2: event-level test and bootstrap with 7-day market episodes

In [8]:
NEP2 = int(EPI2.max()) + 1
rows = []
for k in SYS_ALERT:
    for nname in NULLS:
        x = NR[(k, nname)]
        for m, obs in [('ep2_pre', np.unique(EPI2[OBS[k]['pre']]).size), ('ep2_inp', np.unique(EPI2[OBS[k]['inp']]).size)]:
            v = x['cnt'][m]
            rows.append(dict(system=k, null=nname, statistic=m, observed=int(obs), null_mean=float(v.mean()), null_q05=float(np.percentile(v, 5)),
                             null_q95=float(np.percentile(v, 95)), p_one_sided=(1 + int((v >= obs).sum())) / (len(v) + 1), draws=len(v),
                             events=len(EV), episodes=NEP2, alerts=OBS[k]['alerts']))
T8 = pd.DataFrame(rows)
for m in ['ep2_pre', 'ep2_inp']:
    sel = (T8['null'] == 'budget-preserving') & (T8.statistic == m)
    T8.loc[sel, 'p_holm'] = C.holm(T8.loc[sel, 'p_one_sided'].to_numpy())
T8.to_csv(OUTB / '8_event_level_tests_episodes7.csv', index=False)
display(T8.round(4))

rngB2 = np.random.default_rng(SPEC['event_level']['episode_bootstrap']['seed'])
IDX2 = rngB2.integers(0, NEP2, size=(NBE, NEP2))
rows = []; DIFF2 = {}
for k in SYS_ALERT:
    for nname in NULLS:
        x = NR[(k, nname)]; al = OBS[k]['alerts']
        ex_pre = np.bincount(EPI2, OBS[k]['pre'] - x['p_pre'], minlength=NEP2); ex_in = np.bincount(EPI2, OBS[k]['inp'] - x['p_in'], minlength=NEP2)
        b_pre = 100 * ex_pre[IDX2].sum(1) / al; b_in = 100 * ex_in[IDX2].sum(1) / al; b_d = b_in - b_pre
        for m, pt, b in [('pre', 100 * ex_pre.sum() / al, b_pre), ('inp', 100 * ex_in.sum() / al, b_in), ('inp - pre', 100 * (ex_in.sum() - ex_pre.sum()) / al, b_d)]:
            rows.append(dict(system=k, null=nname, measure=m, excess_per_100_alerts=float(pt), lo95=float(np.percentile(b, 2.5)), hi95=float(np.percentile(b, 97.5)),
                             alerts=al, episodes=NEP2, draws=NBE))
        if nname == 'budget-preserving':
            DIFF2[k] = (float(np.percentile(b_d, 2.5)), float(np.percentile(b_d, 97.5)))
T9 = pd.DataFrame(rows); T9.to_csv(OUTB / '9_effect_sizes_episodes7.csv', index=False)
display(T9[T9['null'] == 'budget-preserving'].round(3))

,system,null,statistic,observed,null_mean,null_q05,null_q95,p_one_sided,draws,events,episodes,alerts,p_holm
0,M0,budget-preserving,ep2_pre,9,22.8522,20.0,26.0,1.0000,5000,241,82,805,1.0000
1,M0,budget-preserving,ep2_inp,34,36.9334,33.0,41.0,0.9316,5000,241,82,805,1.0000
2,M0,joint circular,ep2_pre,9,18.1586,13.0,24.0,0.9988,5000,241,82,805,NaN
3,M0,joint circular,ep2_inp,34,21.5970,17.0,27.0,0.0002,5000,241,82,805,NaN
4,M0,short joint shift,ep2_pre,9,17.2950,11.0,23.0,1.0000,1000,241,82,805,NaN
5,M0,short joint shift,ep2_inp,34,18.8490,16.0,24.0,0.0010,1000,241,82,805,NaN
6,M1,budget-preserving,ep2_pre,14,23.4346,20.0,27.0,1.0000,5000,241,82,823,1.0000
7,M1,budget-preserving,ep2_inp,23,32.8742,29.0,37.0,1.0000,5000,241,82,823,1.0000
8,M1,joint circular,ep2_pre,14,17.0524,13.0,22.0,0.9040,5000,241,82,823,NaN
9,M1,joint circular,ep2_inp,23,20.3694,16.0,25.0,0.2394,5000,241,82,823,NaN


,system,null,measure,excess_per_100_alerts,lo95,hi95,alerts,episodes,draws
0,M0,budget-preserving,pre,-3.078,-6.483,0.136,805,82,10000
1,M0,budget-preserving,inp,1.970,-2.886,7.210,805,82,10000
2,M0,budget-preserving,inp - pre,5.048,-0.568,11.715,805,82,10000
9,M1,budget-preserving,pre,-2.111,-5.822,1.452,823,82,10000
10,M1,budget-preserving,inp,-1.584,-5.565,2.664,823,82,10000
11,M1,budget-preserving,inp - pre,0.526,-4.359,6.366,823,82,10000
18,GBM,budget-preserving,pre,-0.172,-3.132,2.825,713,82,10000
19,GBM,budget-preserving,inp,4.441,-0.480,9.541,713,82,10000
20,GBM,budget-preserving,inp - pre,4.612,-0.958,10.772,713,82,10000


## 6. Summary (descriptive; Q1–Q4 readings as in notebook 38)

In [9]:
h1b = {k: bool(T8[(T8.system == k) & (T8['null'] == 'budget-preserving') & (T8.statistic == 'ep2_inp')].p_holm.iloc[0] < 0.05) for k in SYS_ALERT}
h2b = {k: bool(T8[(T8.system == k) & (T8['null'] == 'budget-preserving') & (T8.statistic == 'ep2_pre')].p_holm.iloc[0] < 0.05) for k in SYS_ALERT}
r7 = T7.set_index(['H', 'negatives', 'system', 'measure'])
SUMB = dict(status='registered secondary outputs computed after the notebook-38 lock; descriptive; Q1-Q4 unchanged',
            notebook38_lock=LOCK38, spec_sha256=SPEC_SHA, reproduction_guard=f'{len(FILES)} files reproduced exactly',
            pooled_D20_M0=dict(S_in=r7.loc[(20, 'main', 'M0', 'S_in')].to_dict(), std=r7.loc[(20, 'main', 'M0', 'std')].to_dict(),
                               pre=r7.loc[(20, 'main', 'M0', 'pre')].to_dict(), inp=r7.loc[(20, 'main', 'M0', 'inp')].to_dict(),
                               reg=r7.loc[(20, 'main', 'M0', 'reg')].to_dict()),
            episodes7=dict(episodes=NEP2, H1_in_progress=h1b, H2_pre_onset=h2b, diff_in_minus_pre_95=DIFF2))
(OUTB / '38b_SUMMARY.json').write_text(json.dumps(SUMB, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))
print(json.dumps(SUMB, indent=2, default=str)); print('done ->', OUTB)

{
  "status": "registered secondary outputs computed after the notebook-38 lock; descriptive; Q1-Q4 unchanged",
  "notebook38_lock": {
    "evaluated_utc": "2026-11-03T09:39:18.018952+00:00",
    "spec_sha256": "994ec994626d023acf90241d3c90c3d4baaa78c930e2cea0a22c00571f3e5481",
    "frozen_sha256": "edab20d2dd110fe93fe2a0623eba1761e88d4460db234718fd528d3ff998e50a"
  },
  "spec_sha256": "994ec994626d023acf90241d3c90c3d4baaa78c930e2cea0a22c00571f3e5481",
  "reproduction_guard": "11 files reproduced exactly",
  "pooled_D20_M0": {
    "S_in": {
      "point": 0.8339848520599643,
      "lo95": 0.7205384831776885,
      "hi95": 0.986081417054123,
      "finite_draws": 2000.0
    },
    "std": {
      "point": 0.7046309205682779,
      "lo95": 0.6672636250007188,
      "hi95": 0.7437582807084303,
      "finite_draws": 2000.0
    },
    "pre": {
      "point": 0.5734608492135562,
      "lo95": 0.5054652785267593,
      "hi95": 0.6341103096438877,
      "finite_draws": 2000.0
    },
    "inp": 